## Phase 1: Exploratory Data Analysis (EDA) and Ground Truth Formulation

In [ ]:
import trimesh
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os
import glob
from tqdm.notebook import tqdm

# Configure plotting aesthetics
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)

# Configuration
DATA_DIR = "../modelnet40"
files = glob.glob(os.path.join(DATA_DIR, "*.off"))
print(f"Found {len(files)} .off files.")

ModuleNotFoundError: No module named 'trimesh'

### Step 1: Geometric Sanity Checks

In [ ]:
def audit_meshes(file_list, sample_size=500):
    audit_data =[]
    
    # Process a representative sample to save time during EDA
    sample_files = np.random.choice(file_list, min(sample_size, len(file_list)), replace=False)
    
    for f in tqdm(sample_files, desc="Auditing Meshes"):
        try:
            # force='mesh' prevents returning complex Scene graphs
            mesh = trimesh.load(f, force='mesh')
            
            audit_data.append({
                'filename': os.path.basename(f),
                'vertices': len(mesh.vertices),
                'faces': len(mesh.faces),
                'is_watertight': mesh.is_watertight,
                'volume': mesh.volume if mesh.is_watertight else np.nan,
                'bounding_box_max': mesh.extents.max(),
                'bounding_box_min': mesh.extents.min(),
                'aspect_ratio': mesh.extents.max() / (mesh.extents.min() + 1e-6)
            })
        except Exception as e:
            print(f"Failed to load {f}: {e}")
            
    return pd.DataFrame(audit_data)

df_audit = audit_meshes(files)

# Display Summary Statistics
display(df_audit.describe())

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

sns.histplot(df_audit['vertices'], bins=50, ax=axes[0], color='blue')
axes[0].set_title('Distribution of Vertex Counts')
axes[0].set_xscale('log') # Log scale because some meshes are incredibly dense

sns.histplot(df_audit['aspect_ratio'], bins=50, ax=axes[1], color='orange')
axes[1].set_title('Aspect Ratio (Max_Extent / Min_Extent)')

sns.countplot(data=df_audit, x='is_watertight', ax=axes[2], palette='Set2')
axes[2].set_title('Watertight vs. Non-Watertight')

plt.tight_layout()
plt.show()

### Mathematical Model

In [ ]:
def calculate_print_cost(mesh, rot_x_deg, rot_y_deg, overhang_angle_threshold=45):
    """
    Calculates the normalized cost of printing a mesh at a given rotation.
    Lower cost = Better orientation.
    """
    # 1. Rotate the mesh mathematically (using a copy to preserve original)
    rotated_mesh = mesh.copy()
    rx, ry = np.radians(rot_x_deg), np.radians(rot_y_deg)
    
    rot_matrix_x = trimesh.transformations.rotation_matrix(rx, [1, 0, 0])
    rot_matrix_y = trimesh.transformations.rotation_matrix(ry,[0, 1, 0])
    transform = trimesh.transformations.concatenate_matrices(rot_matrix_y, rot_matrix_x)
    
    rotated_mesh.apply_transform(transform)
    
    # 2. Metric A: Z-Height (Proxy for print time / slice count)
    z_height = rotated_mesh.extents[2] 
    
    # 3. Metric B: Overhang Area (Proxy for support material)
    # Normals pointing down (negative Z) beyond the threshold require supports
    normals = rotated_mesh.face_normals
    face_areas = rotated_mesh.area_faces
    
    # Z-component of the normal vector
    z_normals = normals[:, 2]
    
    # Threshold in radians
    threshold_rad = np.radians(overhang_angle_threshold)
    # cos(180 - threshold) gives the z-component boundary
    limit_z = -np.cos(threshold_rad) 
    
    # Mask for faces that point downwards past the threshold
    overhang_mask = z_normals < limit_z
    overhang_area = np.sum(face_areas[overhang_mask])
    
    return z_height, overhang_area

def find_optimal_orientation(mesh, search_resolution=10):
    """
    Performs a grid search over X and Y rotations to find the optimal orientation.
    search_resolution: Step size in degrees. Lower is more accurate but slower.
    """
    # Normalize mesh scale first so the optimization landscape is stable
    mesh = mesh.copy()
    mesh.apply_scale(1.0 / mesh.extents.max())
    
    angles_x = np.arange(0, 360, search_resolution)
    angles_y = np.arange(0, 360, search_resolution)
    
    results =[]
    
    # Pre-calculate maximum possible area for normalization
    total_area = mesh.area
    max_possible_z = mesh.extents.max() # Diagonal max
    
    for rx in angles_x:
        for ry in angles_y:
            z_h, ov_a = calculate_print_cost(mesh, rx, ry)
            
            # Normalize metrics to [0, 1]
            norm_z = z_h / max_possible_z
            norm_ov = ov_a / total_area
            
            # Cost Function (Equal weight to height and supports for now)
            # You can adjust alpha and beta based on printer specifics
            alpha, beta = 0.5, 0.5 
            cost = (alpha * norm_z) + (beta * norm_ov)
            
            results.append({
                'rx': rx, 'ry': ry, 
                'z_height': norm_z, 'overhang': norm_ov, 
                'cost': cost
            })
            
    df_results = pd.DataFrame(results)
    
    # Find the minimum cost
    best_row = df_results.loc[df_results['cost'].idxmin()]
    
    return best_row, df_results

# Let's test it on a single mesh
test_mesh = trimesh.load(files[0], force='mesh')
best_orientation, optimization_landscape = find_optimal_orientation(test_mesh, search_resolution=15)

print(f"Optimal Orientation for {os.path.basename(files[0])}:")
print(f"Rot X: {best_orientation['rx']}°, Rot Y: {best_orientation['ry']}°")
print(f"Minimum Cost: {best_orientation['cost']:.4f}")

### Analyzing the Optimization Landscape

In [ ]:
landscape_matrix = optimization_landscape.pivot(index='rx', columns='ry', values='cost')

plt.figure(figsize=(10, 8))
sns.heatmap(landscape_matrix, cmap='viridis')
plt.title(f"Print Cost Landscape for {os.path.basename(files[0])}\n(Darker is Better)")
plt.xlabel("Rotation Y (Degrees)")
plt.ylabel("Rotation X (Degrees)")

# Highlight the absolute minimum
best_rx = best_orientation['rx']
best_ry = best_orientation['ry']

# Find indices for the plot
x_idx = list(landscape_matrix.index).index(best_rx)
y_idx = list(landscape_matrix.columns).index(best_ry)

plt.scatter(y_idx + 0.5, x_idx + 0.5, marker='*', color='red', s=200, label='Global Minimum')
plt.legend()
plt.show()